In [1]:
import pandas as pd

In [2]:
template_df = pd.read_csv('notebooks/jacob/sequencing_plate/dna_plate_template.csv')
template_df.head(2)

,PLATE_NAME,PLATE_BARCODE,WELL_LOCATION,TUBE_BARCODE,SAMPLE_NAME,SEQUENCE_NAME,SEQUENCE,SEQUENCE_TYPE,SEQUENCE_FILE,MASS,...,ext-sequence-principalInvestigatorEmail,ext-material-principalInvestigatorEmail,ext-sequence-bioSafetyLevel,ext-material-bioSafetyLevel,ext-sequence-creator,ext-material-creator,ext-sequence-creatorEmail,ext-material-creatorEmail,ext-sequence-creationDateAtICE,ext-material-creationDateAtICE
0,Plate1,123456789,A1,987654321,Sample1,Sequence1,ATCG,CIRCULAR_DNA,sequence1.gb,1,...,example value,example value,3,3,example value,example value,example value,example value,example value,example value


In [3]:
idt_df = pd.read_excel('notebooks/jacob/sequencing_plate/isaac_sequencing_plate_specs.xlsx')
idt_df.head(2)

,Plate Name,Payment Method,Plate Barcode,Sales Order #,Reference #,Well Position,Sequence Name,Sequence,Manufacturing ID,Measured Molecular Weight,Calculated Molecular Weight,OD260,nmoles,µg,Measured Concentration µM,Final Volume µL,Extinction Coefficient L/(mole·cm),Tm,Well Barcode
0,ID_LevSeqPrimers.F,BB01927950,19498535,20772605,504573988,A01,NB01,CAC AAA GAC ACC GAC AAC TTT CTT CAC CCA AGA CC...,680439448,13303.9,13304,10.81,26.26,349,100.23,262,411500,68,NaN
1,ID_LevSeqPrimers.F,BB01927950,19498535,20772605,504573989,A02,NB02,ACA GAC GAC TAC AAA CGG AAT CGA CAC CCA AGA CC...,680439449,13427.1,13427,12.26,28.40,381,100.00,284,431700,69,NaN


In [10]:
sequencing_reverse_primers = [
    ('RB01', 'AAGAAAGTTGTCGGTGTCTTTGTGCGGTGTGCGAAGTAGGTGC'),
    ('RB02', 'TCGATTCCGTTTGTAGTCGTCTGTCGGTGTGCGAAGTAGGTGC'),
    ('RB03', 'GAGTCTTGTGTCCCAGTTACCAGGCGGTGTGCGAAGTAGGTGC'),
    ('RB04', 'TTCGGATTCTATCGTGTTTCCCTACGGTGTGCGAAGTAGGTGC'),
    ('RB05', 'CTTGTCCAGGGTTTGTGTAACCTTCGGTGTGCGAAGTAGGTGC'),
    ('RB06', 'TTCTCGCAAAGGCAGAAAGTAGTCCGGTGTGCGAAGTAGGTGC'),
    ('RB07', 'GTGTTACCGTGGGAATGAATCCTTCGGTGTGCGAAGTAGGTGC'),
    ('RB08', 'TTCAGGGAACAAACCAAGTTACGTCGGTGTGCGAAGTAGGTGC'),
    ('RB09', 'AACTAGGCACAGCGAGTCTTGGTTCGGTGTGCGAAGTAGGTGC'),
    ('RB10', 'AAGCGTTGAAACCTTTGTCCTCTCCGGTGTGCGAAGTAGGTGC'),
]
sequencing_reverse_primers = [
    ('levseq_' + name, seq, 'P' + '%02d' % (ii + 1))
    for ii, (name, seq) in enumerate(sequencing_reverse_primers)
]

project_primers = [
    ('SK.Seq.F', 'CACCCAAGACCACTCTCCGGGCGAAATTAATACGACTCACTATAGGG'),
    ('SK.Seq.R', 'CGGTGTGCGAAGTAGGTGCGCAGCAGCCAACTCAGCTTCC'),
    ('TY.Seq.F', 'CACCCAAGACCACTCTCCGGCAAGTAGTCAGCTGGAGGAATTGAC'),
    ('TY.Seq.R', 'CGGTGTGCGAAGTAGGTGCGCACGCCGGACATCTCTGG'),
    ('PG.Seq.F', 'CACCCAAGACCACTCTCCGGGGGAAGGGCGATCGGTGC'),
    ('PG.Seq.R', 'CGGTGTGCGAAGTAGGTGCGCTCGGCGGCTTCTAATCCG'),
    ('PW.Seq.F', 'CACCCAAGACCACTCTCCGGGCGCCGCACTGCTCCG'),
    ('PW.Seq.R', 'CGGTGTGCGAAGTAGGTGCCGCCTCTCCCCGCGCG'),
    ('LT.Seq.F', 'CACCCAAGACCACTCTCCGGATGCGTCCGGACCTGGTAACAC'),
    ('LT.Seq.R', 'CGGTGTGCGAAGTAGGTGCCTCAGCTATTGCCGCCGCC'),
    ('ID.Seq.F', 'CACCCAAGACCACTCTCCGGTACATATGGCGCCGACCACCAC'),
    ('ID.Seq.R', 'CGGTGTGCGAAGTAGGTGCCTTACTCGAGTTTGGATCCTTAGTTAAC'),
    ('AP.Seq.F', 'CACCCAAGACCACTCTCCGGGCTCAGTCCTAGGGATTATGCTAG'),
    ('AP.Seq.R', 'CGGTGTGCGAAGTAGGTGCGATGCCTGGAGATCCTTACTCGAGT'),
]
project_primers = [
    ('levseq_' + name, seq, 'P' + '%02d' % (24 - len(project_primers) + ii + 1))
    for ii, (name, seq) in enumerate(project_primers)
]

manual_primers = [
    ('levseq_BBR1.Seq.F', 'whatever', 'N24'),
    ('levseq_BBR1.Seq.R', 'whatever', 'O24'),
]

print(sequencing_reverse_primers[0])
print(sequencing_reverse_primers[-1])
print(project_primers[0])
print(project_primers[-1])

('levseq_RB01', 'AAGAAAGTTGTCGGTGTCTTTGTGCGGTGTGCGAAGTAGGTGC', 'P01')
('levseq_RB10', 'AAGCGTTGAAACCTTTGTCCTCTCCGGTGTGCGAAGTAGGTGC', 'P10')
('levseq_SK.Seq.F', 'CACCCAAGACCACTCTCCGGGCGAAATTAATACGACTCACTATAGGG', 'P11')
('levseq_AP.Seq.R', 'CGGTGTGCGAAGTAGGTGCGATGCCTGGAGATCCTTACTCGAGT', 'P24')


In [11]:
def well_to_row_col(well):
    return well[0], int(well[1:])

def convert_96_well_to_384_well(well, row_offset=0, col_offset=0):
    old_row, old_col = well_to_row_col(well)
    new_row = chr((ord(old_row) - ord('A')) * 2 + ord('A') + row_offset)
    new_col = (old_col - 1) * 2 + 1 + col_offset
    return f'{new_row}{new_col:02d}'

assert convert_96_well_to_384_well('A01') == 'A01'
assert convert_96_well_to_384_well('A03') == 'A05'
assert convert_96_well_to_384_well('B02') == 'C03'
assert convert_96_well_to_384_well('B02', 1, 1) == 'D04'


new_df = pd.DataFrame({
    'WELL_LOCATION': idt_df['Well Position'].apply(convert_96_well_to_384_well),
    'SEQUENCE': idt_df['Sequence'].apply(lambda x: x.replace(' ', '')),
    'SEQUENCE_NAME': idt_df['Sequence Name'].apply(lambda x: 'levseq_' + x),
})

for primer_name, primer_seq, primer_plate_position in sequencing_reverse_primers + project_primers + manual_primers:
    new_df = pd.concat([new_df, pd.DataFrame({
        'WELL_LOCATION': [primer_plate_position],
        'SEQUENCE': [primer_seq],
        'SEQUENCE_NAME': [primer_name],
    })], ignore_index=True)

new_df['VOLUME'] = 50
new_df['VOLUMETRIC_UNIT'] = 'uL'
new_df['SEQUENCE_TYPE'] = 'OLIGO'
new_df['PLATE_NAME'] = 'levseq_primer_plate'
new_df.tail(2)


,WELL_LOCATION,SEQUENCE,SEQUENCE_NAME,VOLUME,VOLUMETRIC_UNIT,SEQUENCE_TYPE,PLATE_NAME
120,N24,whatever,levseq_BBR1.Seq.F,50,uL,OLIGO,levseq_primer_plate
121,O24,whatever,levseq_BBR1.Seq.R,50,uL,OLIGO,levseq_primer_plate


In [12]:
new_df.to_csv('notebooks/jacob/sequencing_plate/levseq_primer_plate.csv', index=False)

# Create echo

In [13]:
new_df.tail(25)

,WELL_LOCATION,SEQUENCE,SEQUENCE_NAME,VOLUME,VOLUMETRIC_UNIT,SEQUENCE_TYPE,PLATE_NAME
97,P02,TCGATTCCGTTTGTAGTCGTCTGTCGGTGTGCGAAGTAGGTGC,levseq_RB02,50,uL,OLIGO,levseq_primer_plate
98,P03,GAGTCTTGTGTCCCAGTTACCAGGCGGTGTGCGAAGTAGGTGC,levseq_RB03,50,uL,OLIGO,levseq_primer_plate
99,P04,TTCGGATTCTATCGTGTTTCCCTACGGTGTGCGAAGTAGGTGC,levseq_RB04,50,uL,OLIGO,levseq_primer_plate
100,P05,CTTGTCCAGGGTTTGTGTAACCTTCGGTGTGCGAAGTAGGTGC,levseq_RB05,50,uL,OLIGO,levseq_primer_plate
101,P06,TTCTCGCAAAGGCAGAAAGTAGTCCGGTGTGCGAAGTAGGTGC,levseq_RB06,50,uL,OLIGO,levseq_primer_plate
102,P07,GTGTTACCGTGGGAATGAATCCTTCGGTGTGCGAAGTAGGTGC,levseq_RB07,50,uL,OLIGO,levseq_primer_plate
103,P08,TTCAGGGAACAAACCAAGTTACGTCGGTGTGCGAAGTAGGTGC,levseq_RB08,50,uL,OLIGO,levseq_primer_plate
104,P09,AACTAGGCACAGCGAGTCTTGGTTCGGTGTGCGAAGTAGGTGC,levseq_RB09,50,uL,OLIGO,levseq_primer_plate
105,P10,AAGCGTTGAAACCTTTGTCCTCTCCGGTGTGCGAAGTAGGTGC,levseq_RB10,50,uL,OLIGO,levseq_primer_plate
106,P11,CACCCAAGACCACTCTCCGGGCGAAATTAATACGACTCACTATAGGG,levseq_SK.Seq.F,50,uL,OLIGO,levseq_primer_plate


In [14]:
source_plate = new_df.copy()

In [15]:
12.5 * 0.25/100

0.03125

In [25]:
# 250nM outer primer
# 50nM inner primer

OUTER_PRIMER_VOLUME = 60  # 60nL gives ~500nM when transfering 100uM soln into 12.5uL
INNER_PRIMER_VOLUME = OUTER_PRIMER_VOLUME / 10.0

def create_transfers_for_user(
    right_outer_primer_name,
    left_inner_primer_name,
    right_inner_primer_name,
    source_plate_name='source_plate_1',
    dest_plate_name='dest_plate_1',
    # dest_row_offset=0,
    # dest_col_offset=0,
):
    transfers = pd.DataFrame()
    for simulated_row in 'ABCDEFGH':
        for simulated_col in range(1, 13):
            left_inner_primer_source_well = convert_96_well_to_384_well(f'{simulated_row}{simulated_col:02d}', 0, 0)

            # dest_well = convert_96_well_to_384_well(f'{simulated_row}{simulated_col:02d}', dest_row_offset, dest_col_offset)
            dest_well = f'{simulated_row}{simulated_col:02d}'

            transfers = pd.concat([transfers, pd.DataFrame({
                'SOURCE_PLATE': [source_plate_name, source_plate_name],
                'SOURCE_WELL': [
                    left_inner_primer_source_well,
                    new_df.loc[new_df['SEQUENCE_NAME'] == right_outer_primer_name, 'WELL_LOCATION'].values[0],
                ],
                'DESTINATION_PLATE': [dest_plate_name, dest_plate_name],
                'DESTINATION_WELL': [dest_well, dest_well],
                'TRANSFER_VOLUME': [OUTER_PRIMER_VOLUME, OUTER_PRIMER_VOLUME],
            })], ignore_index=True)

            transfers = pd.concat([transfers, pd.DataFrame({
                'SOURCE_PLATE': [source_plate_name, source_plate_name],
                'SOURCE_WELL': [
                    new_df.loc[new_df['SEQUENCE_NAME'] == left_inner_primer_name, 'WELL_LOCATION'].values[0],
                    new_df.loc[new_df['SEQUENCE_NAME'] == right_inner_primer_name, 'WELL_LOCATION'].values[0],
                ],
                'DESTINATION_PLATE': [dest_plate_name, dest_plate_name],
                'DESTINATION_WELL': [dest_well, dest_well],
                'TRANSFER_VOLUME': [INNER_PRIMER_VOLUME, INNER_PRIMER_VOLUME],
            })], ignore_index=True)

    return transfers
transfers = pd.concat([
    # create_transfers_for_user('levseq_RB01', 'levseq_ID.Seq.F', 'levseq_ID.Seq.R', 8, 0),
    # create_transfers_for_user('levseq_RB02', 'levseq_LT.Seq.F', 'levseq_LT.Seq.R', 0, 0),
    # create_transfers_for_user('levseq_RB03', 'levseq_AP.Seq.F', 'levseq_AP.Seq.R', 0, 1),
    create_transfers_for_user('levseq_RB01', 'levseq_BBR1.Seq.F', 'levseq_BBR1.Seq.R', dest_plate_name='AP_OplR_R2_levseq_P1'),
    create_transfers_for_user('levseq_RB02', 'levseq_BBR1.Seq.F', 'levseq_BBR1.Seq.R', dest_plate_name='AP_OplR_R2_levseq_P2'),
], ignore_index=True)
transfers.head()


,SOURCE_PLATE,SOURCE_WELL,DESTINATION_PLATE,DESTINATION_WELL,TRANSFER_VOLUME
0,source_plate_1,A01,AP_OplR_R2_levseq_P1,A01,60.0
1,source_plate_1,P01,AP_OplR_R2_levseq_P1,A01,60.0
2,source_plate_1,N24,AP_OplR_R2_levseq_P1,A01,6.0
3,source_plate_1,O24,AP_OplR_R2_levseq_P1,A01,6.0
4,source_plate_1,A03,AP_OplR_R2_levseq_P1,A02,60.0


In [26]:
transfers

,SOURCE_PLATE,SOURCE_WELL,DESTINATION_PLATE,DESTINATION_WELL,TRANSFER_VOLUME
0,source_plate_1,A01,AP_OplR_R2_levseq_P1,A01,60.0
1,source_plate_1,P01,AP_OplR_R2_levseq_P1,A01,60.0
2,source_plate_1,N24,AP_OplR_R2_levseq_P1,A01,6.0
3,source_plate_1,O24,AP_OplR_R2_levseq_P1,A01,6.0
4,source_plate_1,A03,AP_OplR_R2_levseq_P1,A02,60.0
...,...,...,...,...,...
763,source_plate_1,O24,AP_OplR_R2_levseq_P2,H11,6.0
764,source_plate_1,O23,AP_OplR_R2_levseq_P2,H12,60.0
765,source_plate_1,P02,AP_OplR_R2_levseq_P2,H12,60.0
766,source_plate_1,N24,AP_OplR_R2_levseq_P2,H12,6.0


In [28]:
transfers.to_csv('notebooks/jacob/sequencing_plate/echo_two_bbr1_plates.csv', index=False)